# Generate incremental features for planned MIDI files

Run `MIDI_Ingest.ipynb` first. This notebook maintains exactly three new-only CSVs: train, dev, and test. On later runs it reuses rows already present in those CSVs and extracts features only for newly accepted plan entries.


In [1]:
from pathlib import Path
import time

import mido
import numpy as np
import pandas as pd
import pretty_midi
from music21 import chord, converter

SPLIT_PLAN_PATH = Path('../data/split_plans/candidate_split_plan.csv')
DATA_DIR = Path('../data')
ORIGINAL_FEATURE_PATH = DATA_DIR / 'features/original features/train_features.csv'
NEW_FEATURE_PATHS = {split: DATA_DIR / f'features/new_{split}_features.csv' for split in ('train', 'dev', 'test')}
FEATURE_COLUMNS = pd.read_csv(ORIGINAL_FEATURE_PATH, nrows=0).columns.tolist()
KEEP_STATUSES = {'keep_unique', 'keep_candidate_representative'}
GM_FAMILIES = ['piano', 'chromatic_percussion', 'organ', 'guitar', 'bass', 'strings', 'ensemble', 'brass', 'reed', 'pipe', 'synth_lead', 'synth_pad', 'synth_effects', 'ethnic', 'percussive', 'sound_effects']


In [2]:
def normalized_path(value):
    return str(value).replace('/', '\\').casefold()

def feature_key(composer, relative_path):
    return composer.casefold(), normalized_path(relative_path)

def relative_data_path(path):
    return str(path.resolve().relative_to(DATA_DIR.resolve()))

def extract_features(path, composer):
    midi = pretty_midi.PrettyMIDI(str(path))
    note_tracks = [instrument for instrument in midi.instruments if instrument.notes]
    notes = [note for instrument in note_tracks for note in instrument.notes]
    if not notes:
        raise ValueError('No notes found')
    pitches = np.array([note.pitch for note in notes], dtype=float)
    velocities = np.array([note.velocity for note in notes], dtype=float)
    durations = np.array([note.end - note.start for note in notes], dtype=float)
    onsets = np.sort(np.array([note.start for note in notes], dtype=float))
    total_duration = midi.get_end_time()
    chord_count = sum(1 for value in converter.parse(str(path)).chordify().recurse().getElementsByClass(chord.Chord))
    pitch_hist = midi.get_pitch_class_histogram()
    pitch_probabilities = pitch_hist / (pitch_hist.sum() + 1e-10)
    events = sorted([(note.start, 1) for note in notes] + [(note.end, -1) for note in notes], key=lambda event: event[0])
    active_notes = max_polyphony = event_index = 0
    weighted_polyphony, previous_time = 0.0, events[0][0]
    while event_index < len(events):
        event_time = events[event_index][0]
        weighted_polyphony += active_notes * (event_time - previous_time)
        while event_index < len(events) and events[event_index][0] == event_time:
            active_notes += events[event_index][1]
            event_index += 1
        max_polyphony = max(max_polyphony, active_notes)
        previous_time = event_time
    tempo = midi.estimate_tempo()
    features = {'composer': composer, 'filename': path.name, 'tempo': tempo, 'num_notes': len(notes),
        'num_chords': chord_count, 'avg_pitch': np.mean(pitches), 'pitch_range': np.ptp(pitches),
        'avg_duration': np.mean(durations), 'avg_velocity': np.mean(velocities), 'note_density': len(notes) / total_duration,
        'pitch_entropy': -np.sum(pitch_probabilities * np.log(pitch_probabilities + 1e-10)), 'pitch_class_variance': np.var(pitch_hist),
        'range_normalized': np.ptp(pitches) / (np.mean(pitches) + 1e-6), 'notes_per_chord': len(notes) / (chord_count + 1),
        'chord_density': chord_count / (len(notes) + 1), 'velocity_variation': np.mean(velocities) / (tempo + 1),
        'tempo_note_ratio': tempo / (len(notes) + 1), 'chromatic_ratio': pitch_hist[[1, 3, 6, 8, 10]].sum() / (pitch_hist.sum() + 1e-6),
        'relative_path': relative_data_path(path), 'total_duration': total_duration,
        'num_midi_tracks': len(mido.MidiFile(str(path)).tracks), 'num_note_tracks': len(note_tracks),
        'num_unique_programs': len({instrument.program for instrument in note_tracks if not instrument.is_drum}),
        'drum_track_count': sum(instrument.is_drum for instrument in note_tracks), 'has_drums': int(any(instrument.is_drum for instrument in note_tracks)),
        'pitch_std': np.std(pitches), 'pitch_median': np.median(pitches), 'velocity_std': np.std(velocities),
        'velocity_range': np.ptp(velocities), 'duration_std': np.std(durations), 'duration_median': np.median(durations),
        'onset_interval_mean': np.mean(np.diff(onsets)) if len(onsets) > 1 else 0.0,
        'onset_interval_std': np.std(np.diff(onsets)) if len(onsets) > 1 else 0.0,
        'max_polyphony': max_polyphony, 'avg_polyphony': weighted_polyphony / total_duration if total_duration else 0.0}
    for index, value in enumerate(pitch_hist): features[f'pitch_class_{index}'] = value
    for family in GM_FAMILIES: features[f'gm_{family}_track_count'] = 0
    for instrument in note_tracks:
        if not instrument.is_drum: features[f'gm_{GM_FAMILIES[instrument.program // 8]}_track_count'] += 1
    return features


In [3]:
if not SPLIT_PLAN_PATH.is_file():
    raise FileNotFoundError(f'Run MIDI_Ingest.ipynb first: {SPLIT_PLAN_PATH}')

plan = pd.read_csv(SPLIT_PLAN_PATH)
keepable = plan.loc[plan['status'].isin(KEEP_STATUSES)].copy()
keepable['relative_path'] = keepable['path'].map(lambda value: relative_data_path(Path(value)))
keepable['feature_key'] = [feature_key(composer, path) for composer, path in zip(keepable['composer'], keepable['relative_path'])]
if keepable['feature_key'].duplicated().any():
    raise ValueError('The plan contains duplicate keepable composer/path entries.')

cached_rows = {}
for split, output_path in NEW_FEATURE_PATHS.items():
    if not output_path.is_file():
        continue
    existing = pd.read_csv(output_path)
    if existing.columns.tolist() != FEATURE_COLUMNS:
        raise ValueError(f'Unexpected schema in {output_path}; expected the original feature columns.')
    for row in existing.to_dict('records'):
        cached_rows[feature_key(row['composer'], row['relative_path'])] = row

rows_by_split = {split: [] for split in NEW_FEATURE_PATHS}
missing = []
for row in keepable.itertuples(index=False):
    split = row.recommended_split
    cached = cached_rows.get(row.feature_key)
    if cached is not None:
        rows_by_split[split].append(cached)
    else:
        missing.append(row)

print(f'Keeping {len(keepable):,} planned rows; reusing {len(keepable) - len(missing):,}; extracting {len(missing):,} new rows.')


Keeping 611 planned rows; reusing 548; extracting 63 new rows.


In [4]:
errors = []
for index, row in enumerate(missing, start=1):
    path = Path(row.path)
    try:
        rows_by_split[row.recommended_split].append(extract_features(path, row.composer))
    except Exception as error:
        errors.append({'path': row.path, 'composer': row.composer, 'error': str(error)})
    if index % 10 == 0 or index == len(missing):
        print(f'Processed {index:,}/{len(missing):,} new MIDI files.')

error_report = pd.DataFrame(errors)
display(error_report)
if not error_report.empty:
    raise RuntimeError(f'Feature extraction failed for {len(error_report)} files; existing CSVs were not changed.')

for split, output_path in NEW_FEATURE_PATHS.items():
    frame = pd.DataFrame(rows_by_split[split]).reindex(columns=FEATURE_COLUMNS)
    frame = frame.sort_values(['composer', 'relative_path'], ignore_index=True)
    frame.to_csv(output_path, index=False)
    print(f'Saved {len(frame):,} rows to {output_path}')


c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\pretty_midi\pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(
c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'F\xfcr Elise'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)
c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=3, data=b'Beethoven F\xfcr Elise'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)
c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages

Processed 10/63 new MIDI files.


c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Piano Sonata No. 8 (Beethoven), Op. 13, "Path\xe9tique", 1st movement'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)
c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Piano Sonata No. 8 (Beethoven), Op. 13, "Path\xe9tique", 2nd movement.'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)
c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=0, data=b'Piano Sonata No. 8 (Beethoven), Op. 13, "P

Processed 20/63 new MIDI files.


c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=4, data=b'Copyright \xa9 2002 by Bernd Kr\xfcger'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)


...

Processed 40/63 new MIDI files.


c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2006 by Bernd Krueger'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)
c:\Users\Maxtw\OneDrive\Desktop\511 Project\.venv\Lib\site-packages\music21\midi\translate.py:2008: TranslateWarning: Unable to determine instrument from <music21.midi.MidiEvent SEQUENCE_TRACK_NAME, track=5, data=b'Copyright \xa9 2005 by Bernd Kr\xfcger'>; getting generic Instrument
  metaObj = midiEventToInstrument(e, encoding=encoding)


Processed 50/63 new MIDI files.
Processed 60/63 new MIDI files.
Processed 63/63 new MIDI files.


""


Saved 412 rows to ..\data\features\new_train_features.csv
Saved 109 rows to ..\data\features\new_dev_features.csv
Saved 90 rows to ..\data\features\new_test_features.csv
